# Rubin DR1-like SN Ia Simulation

This notebook simulates a DR1-like sample of Type Ia supernovae using [sncosmo](https://sncosmo.readthedocs.io/en/stable/)'s SALT3-NIR model and a Rubin OpSim (simulated survey). It scales up the [Rubin DP2 notebook](jem-rubin_dp2.ipynb): the noise model and fitting are the same, but the survey is the first year of the LSST Wide-Fast-Deep (WFD) survey and the redshifts follow the SN Ia volumetric rate out to z = 1.

The main differences from DP2 are:
* **Survey:** an OpSim (one row per visit) instead of the DP2 visit-detector table (one row per CCD). Each visit is treated as a 1.75 degree radius circle, so no detector footprint is needed.
* **Window:** the first year of the survey, WFD visits only (no deep drilling fields).
* **Redshifts:** drawn from the volumetric rate, so the simulated sample is a random subsample of the full year-one population. We keep track of the scale factor between the two.
* **SN Ia model:** SALT3-NIR ([Pierel et al. 2022](https://scixplorer.org/abs/2022ApJ...939...11P/abstract)) instead of `salt2-h17`, with x1, c, alpha, and beta updated to representative values from the DES-SN5YR reanalysis ([Popovic et al. 2026a](https://scixplorer.org/abs/2026MNRAS.548ag632P/abstract)). The model is set by `SALT_MODEL` and is used for both the simulation and the fits.
* **Fitting:** many more objects to fit, so the fits run in parallel using the helper functions in [salt_fitting.py](salt_fitting.py).

## Configuration

All of the settings we may want to change are collected here. `N_SIM` sets the size of the subsample. The full rate-normalized population for year one is computed below and is a few million objects.

In [ ]:
import numpy as np

JEM_LCL_BASE_DR = "/Users/j.mccleary/Software/LightCurveLynx/"
OPSIM_FILE = f"{JEM_LCL_BASE_DR}/obstable/baseline_v5.3.7_10yrs.db"
OUTPUT_DIR = f"{JEM_LCL_BASE_DR}/sims/dr1_snia"

# The sncosmo SN Ia model, used both to simulate and to fit the light curves.
SALT_MODEL = "salt3-nir"

N_SIM = 50_000  # The number of objects to simulate (a subsample of the full population).
SEED = 42

SURVEY_DAYS = 365.25  # Length of the survey window (the first year).
Z_MIN, Z_MAX = 0.01, 1.0

# Cosmology used for both the rates and the distance moduli.
H0, OMEGA_M = 73.0, 0.3

# The rest-frame window of the light curves (same as DP2).
REST_WINDOW = (-20.0, 50.0)

# t0 is sampled from slightly outside the survey window, so that objects which peak just before
# the survey starts (or just after it ends) still contribute their partial light curves.
# At z = 1, the rest-frame window above covers observer-frame [-40, +100] days.
T0_PAD_BEFORE = REST_WINDOW[1] * (1.0 + Z_MAX)  # 100 days
T0_PAD_AFTER = -REST_WINDOW[0] * (1.0 + Z_MAX)  # 40 days

## Load the first year of WFD visits

We load the OpSim with an SQL query that selects only the first year of WFD visits. In the v5.x baselines each visit has a `target_name` that records which part of the survey footprint it belongs to. `lowdust` is the extragalactic WFD region, and the others are the Galactic plane (`dusty_plane`, `bulgy`), the North Ecliptic Spur (`nes`), the South Celestial Pole (`scp`), and the Magellanic Clouds. We also drop deep drilling field visits (whose `scheduler_note` starts with `DD:`) and target-of-opportunity visits (`ToO`), even when they fall inside the WFD region.

Selecting in SQL also avoids `filter_rows()`, which modifies the `ObsTable` **in place** (and returns the same object) rather than returning a copy.

Note that much of the first year is spent building templates (`scheduler_note` starting with `templates`). These are ordinary visits and we keep them.

In [ ]:
import sqlite3

from lightcurvelynx.obstable.opsim import OpSim

with sqlite3.connect(OPSIM_FILE) as con:
    t_start = con.execute("SELECT MIN(observationStartMJD) FROM observations").fetchone()[0]
t_end = t_start + SURVEY_DAYS

wfd_query = f"""
SELECT * FROM observations
WHERE observationStartMJD < {t_end}
  AND target_name LIKE '%lowdust%'
  AND scheduler_note NOT LIKE 'DD%'
  AND scheduler_note NOT LIKE 'ToO%'
"""
obs_table = OpSim.from_db(OPSIM_FILE, sql_query=wfd_query)

print(f"Loaded {len(obs_table)} WFD visits between MJD {t_start:.1f} and {t_end:.1f}")
print("Visits per filter:", dict(zip(*np.unique(obs_table["filter"], return_counts=True))))

We create the rest of the survey information by using the standard LSST passbands and noise model (Poisson), as in DP2. The OpSim provides everything the noise model needs: the zero points are derived from the airmass, exposure time, and filter; the sky background from the sky brightness; and the PSF size from the seeing. The camera properties (read noise, dark current, pixel scale) are LSSTCam defaults stored in the `OpSim` class.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.survey_info import SurveyInfo

passband_group = PassbandGroup.from_preset(preset="LSST")
survey_info = SurveyInfo(obstable=obs_table, passbands=passband_group, survey_name="LSST")

## The rate-normalized redshift distribution

Instead of sampling the redshift uniformly, we sample it from the number of SNe Ia expected per unit redshift:

$$\frac{dN}{dz} \propto r_v(z) \, \frac{dV_c}{dz} \, \frac{1}{1+z}$$

where $r_v(z) = 2.27 \times 10^{-5} (1+z)^{1.7}$ SNe yr$^{-1}$ Mpc$^{-3}$ $h_{70}^3$ is the volumetric rate from [Frohmaier et al. (2019)](https://ui.adsabs.harvard.edu/abs/2019MNRAS.486.2308F), $dV_c/dz$ is the comoving volume per unit redshift, and the $1/(1+z)$ accounts for cosmological time dilation. These are the defaults in LightCurveLynx's `snia_volumetric_rates()` and `num_snia_per_redshift_bin()`.

To get the total number of SNe in the survey, we multiply by the area of the WFD footprint and the length of the window over which t0 is sampled (the survey year plus the padding on either side). The footprint area comes from the MOC of the visits, so it includes the 1.75 degree circle around each pointing at the edges of the footprint.

In [ ]:
from lightcurvelynx.astro_utils.snia_utils import num_snia_per_redshift_bin

moc = obs_table.build_moc(max_depth=10)
area_deg2 = moc.sky_fraction * 41253.0
solid_angle = area_deg2 * (np.pi / 180.0) ** 2

t0_low, t0_high = t_start - T0_PAD_BEFORE, t_end + T0_PAD_AFTER
t0_years = (t0_high - t0_low) / 365.25

n_per_bin, z_bins = num_snia_per_redshift_bin(
    zmin=Z_MIN, zmax=Z_MAX, znbins=99, solid_angle=solid_angle, H0=H0, Omega_m=OMEGA_M
)
n_total = n_per_bin.sum() * t0_years
scale_factor = n_total / N_SIM

print(f"WFD footprint area: {area_deg2:.0f} deg^2")
print(f"t0 window: {t0_high - t0_low:.0f} days ({t0_years:.2f} years)")
print(f"Expected number of SNe Ia with {Z_MIN} < z < {Z_MAX}: {n_total:,.0f}")
print(f"Simulating {N_SIM:,} objects, so each one represents {scale_factor:.1f} real SNe.")

We use `SamplePDF` to draw redshifts from this distribution. It only needs a function proportional to the PDF, so we do not have to normalize it.

In [ ]:
from astropy.cosmology import FlatLambdaCDM

from lightcurvelynx.astro_utils.snia_utils import snia_volumetric_rates
from lightcurvelynx.math_nodes.scipy_random import SamplePDF

cosmo = FlatLambdaCDM(H0=H0, Om0=OMEGA_M)


def snia_dndz(z):
    """The (unnormalized) number of SNe Ia per unit redshift per year of observer time."""
    return snia_volumetric_rates(z) * cosmo.differential_comoving_volume(z).value / (1.0 + z)


redshift_sampler = SamplePDF(snia_dndz, domain=(Z_MIN, Z_MAX))

## Create the model

Positions are sampled uniformly over the WFD footprint, as in DP2. The x1 and c distributions and the Tripp alpha and beta are updated from the DP2 placeholder values to representative values from the DES-SN5YR reanalysis. The absolute magnitude distribution is unchanged.

The time window for t0 covers the survey year plus the padding described above.

In [ ]:
from lightcurvelynx.astro_utils.snia_utils import DistModFromRedshift, X0FromDistMod
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import ApproximateMOCSampler

# Sample position (RA, dec) uniformly from the footprint of the survey. This sampler
# will take multiple seconds to build as it creates a MOC of the survey footprint.
coverage_sampler = ApproximateMOCSampler.from_obstable(obs_table, depth=10)

# Compute the distance modulus from the redshift, using the same cosmology as the rates.
distmod_func = DistModFromRedshift(redshift_sampler, H0=H0, Omega_m=OMEGA_M)

# Population / Tripp parameters (also used below to propagate fit covariance).
ALPHA, BETA, M0, SIGMA_INT = 0.17, 3.12, -19.3, 0.10

# Sample x1, c, and m_abs using more representative distributions for x1 and c.
x1_func = NumpyRandomFunc("normal", loc=0, scale=1.0)
c_func = NumpyRandomFunc("normal", loc=0, scale=0.1)
m_abs_func = NumpyRandomFunc("normal", loc=M0, scale=SIGMA_INT)

# Compute x0 from the other parameters using the standard Tripp formula.
# alpha=0.17, beta=3.12 are from DESY5 SN Ia cosmology reanalysis (B. Popovic et al 2026)
x0_func = X0FromDistMod(
    distmod=distmod_func,
    x1=x1_func,
    c=c_func,
    alpha=ALPHA,
    beta=BETA,
    m_abs=m_abs_func,
)

# t0 is sampled uniformly over the (padded) survey window.
t0_func = NumpyRandomFunc("uniform", low=t0_low, high=t0_high)

We then define the model using the `SncosmoWrapperModel` class, with the same `LinearDecay` time extrapolation as DP2.

At the higher redshifts in this sample we also need a **wavelength** extrapolation. SALT3-NIR (like SALT2) is only defined above 2000 Å in the rest frame, and at z ≳ 0.5 the blue edge of the u band falls below that. We use `ZeroPadding`, which sets the flux to zero outside the model's wavelength range. The fitting code below only uses bands whose rest-frame effective wavelength is above 2800 Å (see below), so these partially-covered bands do not affect the fits.

In [ ]:
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.utils.extrapolate import LinearDecay, ZeroPadding

source = SncosmoWrapperModel(
    SALT_MODEL,
    t0=t0_func,
    x0=x0_func,
    x1=x1_func,
    c=c_func,
    ra=coverage_sampler.ra,
    dec=coverage_sampler.dec,
    redshift=redshift_sampler,
    node_label="source",
    time_extrapolation=LinearDecay(50.0),  # Linear decay to zero over 50 days past the model's phase range.
    wave_extrapolation=ZeroPadding(),  # Zero flux outside the model's wavelength range.
)

## Generate the simulations

We restrict the simulations to the same rest-frame window as DP2. For 50,000 objects this takes under a minute.

We save the results (including the `params` column) to a parquet file so the fits can be rerun later without resimulating.

In [ ]:
from pathlib import Path

from lightcurvelynx.simulate import simulate_lightcurves

lightcurves = simulate_lightcurves(
    source,
    N_SIM,
    survey_info,
    rest_time_window_offset=REST_WINDOW,
    rng=np.random.default_rng(SEED),
)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
lightcurves.to_parquet(f"{OUTPUT_DIR}/lightcurves_{SALT_MODEL}_n{N_SIM}_seed{SEED}.parquet")

# Keep the parameters column separately for the plots and fits below.
params_col = lightcurves["params"].copy()
lightcurves = lightcurves.drop(columns=["params"])
lightcurves.head()

As a check, the simulated redshifts should follow the rate-normalized distribution. The histogram also shows how many of the objects were observed at all. Objects can fall in gaps between pointings (we sample from an approximate MOC), or have t0 so far from any visit that nothing lands in the rest-frame window.

In [ ]:
import matplotlib.pyplot as plt

true_z = lightcurves["z"].to_numpy()
observed = lightcurves["nobs"].to_numpy() > 0

bins = np.linspace(Z_MIN, Z_MAX, 40)
expected = snia_dndz(0.5 * (bins[1:] + bins[:-1])) * np.diff(bins)
expected *= N_SIM / expected.sum()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(true_z, bins=bins, histtype="step", color="black", label="simulated")
ax.hist(true_z[observed], bins=bins, histtype="step", color="tab:blue", label="observed (nobs > 0)")
ax.stairs(expected, bins, color="tab:red", ls="--", label="expected from rates")
ax.set_xlabel("redshift")
ax.set_ylabel("objects per bin")
ax.legend()
plt.show()

print(f"{observed.sum():,} of {N_SIM:,} objects ({observed.mean():.1%}) have at least one observation.")

We plot the first 3 results that contain enough sampled points to be interesting.

In [ ]:
from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve
from lightcurvelynx.utils.plotting import plot_lightcurves

curr_idx = 0
count = 0
while count < 3:
    result_row = lightcurves.iloc[curr_idx]
    params_dict = params_col.iloc[curr_idx]
    curr_idx += 1

    # If we didn't observe the object at all, skip it.
    current_lc = result_row["lightcurve"]
    if current_lc is None or len(current_lc) == 0:
        continue

    lc_filters = np.asarray(current_lc["filter"], dtype=str)
    lc_mjd = np.asarray(current_lc["mjd"], dtype=float)
    lc_flux = np.asarray(current_lc["flux"], dtype=float)
    lc_fluxerr = np.asarray(current_lc["fluxerr"], dtype=float)

    # If the light curve has only a few observations, covers a very short time span, or
    # is never detected, skip it.
    if len(lc_mjd) < 10 or (np.max(lc_mjd) - np.min(lc_mjd)) < 50 or np.max(lc_flux / lc_fluxerr) < 10:
        continue

    noise_free_lcs = compute_single_noise_free_lightcurve(
        source,
        GraphState.from_dict(params_dict),
        passband_group,
        rest_frame_phase_min=REST_WINDOW[0],
        rest_frame_phase_max=REST_WINDOW[1],
        rest_frame_phase_step=0.5,
    )

    redshift = params_dict["source.redshift"]
    t0 = params_dict["source.t0"]

    plot_lightcurves(
        fluxes=lc_flux,
        times=lc_mjd,
        fluxerrs=lc_fluxerr,
        filters=lc_filters,
        underlying_model=noise_free_lcs,
        title=f"Simulated SNIa Light Curve (redshift={redshift:.4f}, t0={t0:.1f})",
    )
    count += 1

## Fit the simulated light curves with SALT3-NIR

The fitting follows DP2 exactly, using the same model as the simulation (`SALT_MODEL`): LightCurveLynx's LSST passbands registered with sncosmo, an AB zero point of 31.4 (nJy), the redshift held fixed at its true value, `t0`, `x0`, `x1`, and `c` fit with the same bounds and phase range, and bands outside the model's wavelength range dropped. Objects with fewer than 5 usable points, or no point with S/N > 5, are not fit.

One change from DP2: following Mitra et al. (2025), we only fit bands whose rest-frame effective wavelength, λ_eff / (1 + z), is between 2800 and 17,000 Å, the range where SALT3 is well trained. For LSST this drops u above z ≈ 0.33 and g above z ≈ 0.72.

The code has moved into [salt_fitting.py](salt_fitting.py) so that the fits can run in parallel. Each worker process registers the passbands with sncosmo when it starts. Unlike DP2, we fit every detected object instead of the first 200.
`sncosmo.fit_lc()` also returns a covariance matrix for the varied parameters. The imported `salt_fitting_cov` helper stores its 10 independent elements as numeric columns `cov_<parameter>_<parameter>` (parameter order: t0, x0, x1, c). The next section propagates the x0/x1/c block to a standardized distance modulus, conditional on known redshift and fixed Tripp coefficients. This is the covariance of *fitted parameters*, not `modelcov=True` (which includes SALT3 template/model uncertainty in the fitting likelihood).


In [ ]:
import importlib
import os
from concurrent.futures import ProcessPoolExecutor

import pandas as pd

import salt_fitting_cov as salt_fitting

# Reload in case salt_fitting.py was edited since the kernel started (a plain import would keep the old version).
importlib.reload(salt_fitting)

band_names = salt_fitting.register_passbands(passband_group)

fit_tasks = [
    (idx, lightcurves.iloc[idx]["lightcurve"], params_col.iloc[idx], SALT_MODEL)
    for idx in np.flatnonzero(observed)
]
with ProcessPoolExecutor(max_workers=os.cpu_count(), initializer=salt_fitting.register_passbands) as executor:
    fit_rows = [row for row in executor.map(salt_fitting.fit_one, fit_tasks, chunksize=100) if row is not None]


## The DR1-like sample

Scaling by the subsampling factor gives the predicted number of SNe Ia in year one. This depends strongly on the quality cuts. We use the light curve cuts from Mitra et al. (2025):
1. at least three bands with maximum S/N > 4 (among the points used in the fit);
2. |x1| < 3;
3. |c| < 0.3;
4. σ_x1 < 1;
5. σ_t0 < 2 days;
8. 0.01 < z < 1.4 (applied to the true redshift, so it has no effect here);
9. only fit bands with 2800 < λ_eff / (1 + z) < 17,000 Å (applied during the fit, above).

Their cuts 6 and 7 (photometric classification and spectroscopic redshift) do not apply to this simulation. We also require the fit to converge. For comparison, we also show the looser DP2 cuts (at least 5 points near peak, one S/N > 3 detection on either side of peak, and reduced χ² < 3).
The fit catalog also contains `fit_mB`, `fit_mu`, `var_mu_fit`, `err_mu_fit`, and `err_mu_total` (the last adds the configured population intrinsic scatter in quadrature). Note that `err_mu_fit` is a local Gaussian approximation, not the full sampled posterior. The selected-only fits are written to a separate Parquet file.


In [ ]:
fit_results = salt_fitting.apply_quality_cuts(pd.DataFrame(fit_rows).set_index("idx"))

# Convert the fitted SALT3 amplitude to apparent B-band magnitude, then
# propagate the 3x3 (x0, x1, c) block of the fit covariance to Tripp mu.
# redshift, ALPHA, BETA, and M0 are held fixed in this approximation.
fit_results["fit_mB"] = -2.5 * np.log10(fit_results["fit_x0"].where(fit_results["fit_x0"] > 0)) + 10.635
fit_results["fit_mu"] = (
    fit_results["fit_mB"] + ALPHA * fit_results["fit_x1"]
    - BETA * fit_results["fit_c"] - M0
)

g0 = -2.5 / (np.log(10.0) * fit_results["fit_x0"].where(fit_results["fit_x0"] > 0))
mu_var = (
    g0**2 * fit_results["cov_x0_x0"]
    + ALPHA**2 * fit_results["cov_x1_x1"]
    + BETA**2 * fit_results["cov_c_c"]
    + 2.0 * g0 * ALPHA * fit_results["cov_x0_x1"]
    - 2.0 * g0 * BETA * fit_results["cov_x0_c"]
    - 2.0 * ALPHA * BETA * fit_results["cov_x1_c"]
)
fit_results["var_mu_fit"] = mu_var
# Negative variance estimates indicate an invalid covariance; keep as NaN.
fit_results["err_mu_fit"] = np.sqrt(mu_var.where(mu_var >= 0.0))
# Include intrinsic gray scatter only when constructing the cosmology likelihood.
fit_results["err_mu_total"] = np.sqrt(mu_var.where(mu_var >= 0.0) + SIGMA_INT**2)

fit_results.to_parquet(f"{OUTPUT_DIR}/fits_{SALT_MODEL}_n{N_SIM}_seed{SEED}.parquet")
good_fits = fit_results[fit_results["passes_cuts"]]
# A selected-only catalog suitable for sharing in a BLIND run: omit the
# injected true SALT parameters. The private all-fits file above retains them.
selected_public = good_fits.drop(columns=[c for c in good_fits if c.startswith("true_")])
selected_public.to_parquet(f"{OUTPUT_DIR}/selected_fits_{SALT_MODEL}_n{N_SIM}_seed{SEED}.parquet")

print(f"Fit {len(fit_results):,} of the {observed.sum():,} observed objects (the rest were not detected).")
print(f"{len(good_fits):,} pass the Mitra et al. (2025) quality cuts.")
print(f"{fit_results['passes_dp2_cuts'].sum():,} pass the looser DP2 cuts (for comparison).")
print(f"Fit covariances available: {fit_results['covariance_available'].sum():,} / {len(fit_results):,}")
fit_results.head()


In [ ]:
stages = {
    "simulated": np.ones(N_SIM, dtype=bool),
    "observed (nobs > 0)": observed,
    "detected and fit": np.isin(np.arange(N_SIM), fit_results.index),
    "passes DP2 cuts": np.isin(np.arange(N_SIM), fit_results.index[fit_results["passes_dp2_cuts"]]),
    "passes Mitra cuts": np.isin(np.arange(N_SIM), good_fits.index),
}
for name, mask in stages.items():
    print(f"{name:>20s}: {mask.sum():>7,d} simulated -> {mask.sum() * scale_factor:>11,.0f} expected in year one")

fig, ax = plt.subplots(figsize=(6, 4))
for name, mask in stages.items():
    ax.hist(true_z[mask], bins=bins, histtype="step", label=name, weights=np.full(mask.sum(), scale_factor))
ax.set_yscale("log")
ax.set_xlabel("redshift")
ax.set_ylabel("expected SNe Ia per bin (year one)")
ax.legend(fontsize="small")
plt.show()

We plot the fitted SALT model (solid) and the true model (dashed) on top of the noisy data for the first few objects that pass the quality cuts.

In [ ]:
import sncosmo

for idx in good_fits.index[:3]:
    params_dict = params_col.iloc[idx]
    fit_model = sncosmo.Model(source=SALT_MODEL)
    data = salt_fitting.prepare_data(lightcurves.iloc[idx]["lightcurve"], params_dict["source.redshift"], fit_model)
    fit_model.set(**{name: good_fits.loc[idx, f"fit_{name}"] for name in salt_fitting.FIT_PARAMS})

    true_model = sncosmo.Model(source=SALT_MODEL)
    true_model.set(z=params_dict["source.redshift"], **{name: params_dict[f"source.{name}"] for name in salt_fitting.FIT_PARAMS})
    sncosmo.plot_lc(
        data,
        model=[fit_model, true_model],
        model_label=[f"{SALT_MODEL} fit", "Truth"],
        zp=salt_fitting.ZP_NJY,
        figtext=f"object {idx}, z = {params_dict['source.redshift']:.3f}",
    )

Finally, we compare the fitted parameters to the true values for the objects passing the cuts. x0 is shown as the equivalent magnitude offset, -2.5 log10(x0_fit / x0_true).

The top row shows the residuals as a function of redshift, with the median and 16-84th percentile range in redshift bins. The axes are linear: a symmetric-log axis (as in DP2) makes any distribution wider than its linear threshold look bimodal, because it stretches the region around zero where most of the points are.

The bottom row shows the pulls, (fit - truth) / error. If the fit uncertainties are correct, these follow a unit Gaussian (black curve) with a width of 1.

In [ ]:
# Compare the fitted parameters to the true values for the objects passing the cuts. The top row shows the
# residuals as a function of redshift, with the median and 16-84th percentile range in redshift bins. The
# y-axes span four times the overall 16-84th percentile range; the number of points outside is noted in each panel.
# The bottom row shows the pulls, (fit - truth) / error, which should follow a unit Gaussian (black curve) if
# the fit uncertainties are correct. Pulls beyond +/-5 are shown in the end bins.
good = fit_results[fit_results["passes_cuts"]]
mag_per_dex = 2.5 / np.log(10.0)
residuals = {
    "t0 (days)": (good["fit_t0"] - good["true_t0"], good["err_t0"]),
    "x0 (mag)": (-2.5 * np.log10(good["fit_x0"] / good["true_x0"]), mag_per_dex * good["err_x0"] / good["fit_x0"]),
    "x1": (good["fit_x1"] - good["true_x1"], good["err_x1"]),
    "c": (good["fit_c"] - good["true_c"], good["err_c"]),
}
redshift = good["redshift"].to_numpy()
z_edges = np.linspace(Z_MIN, Z_MAX, 11)
z_centers = 0.5 * (z_edges[1:] + z_edges[:-1])
z_bin = np.digitize(redshift, z_edges) - 1
pull_grid = np.linspace(-5, 5, 201)

fig, axes = plt.subplots(2, len(residuals), figsize=(4 * len(residuals), 7))
for col, (label, (resid, err)) in enumerate(residuals.items()):
    resid = resid.to_numpy()
    pull = resid / err.to_numpy()

    # Residuals vs. redshift.
    ax = axes[0, col]
    q16, q84 = np.nanpercentile(resid, [16, 84])
    half_range = 4.0 * max(abs(q16), abs(q84))
    n_outside = np.sum(np.abs(resid) > half_range)
    ax.scatter(redshift, resid, s=2, color="tab:blue", alpha=0.1, rasterized=True)
    pct = np.array([np.nanpercentile(resid[z_bin == b], [16, 50, 84]) for b in range(len(z_centers))])
    ax.fill_between(z_centers, pct[:, 0], pct[:, 2], color="tab:orange", alpha=0.35, lw=0, label="16-84%")
    ax.plot(z_centers, pct[:, 1], color="tab:orange", lw=2, label="median")
    ax.axhline(0.0, color="black", lw=0.8, ls=":")
    ax.set_ylim(-half_range, half_range)
    ax.set_xlabel("redshift")
    ax.set_title(f"fit - truth: {label}")
    ax.text(0.03, 0.03, f"{n_outside} outside range", transform=ax.transAxes, fontsize="small", color="dimgray")

    # Pull distribution.
    ax = axes[1, col]
    q16, q50, q84 = np.nanpercentile(pull, [16, 50, 84])
    ax.hist(np.clip(pull, -5, 5), bins=np.linspace(-5, 5, 61), density=True, color="tab:blue", alpha=0.7)
    ax.plot(pull_grid, np.exp(-0.5 * pull_grid**2) / np.sqrt(2.0 * np.pi), color="black", lw=1.5)
    ax.set_xlim(-5, 5)
    ax.set_xlabel(f"pull: {label}")
    ax.text(
        0.03, 0.95, f"median {q50:+.2f}\nwidth {0.5 * (q84 - q16):.2f}",
        transform=ax.transAxes, va="top", fontsize="small",
    )
axes[0, 0].legend(loc="upper right", fontsize="small")
axes[1, 0].set_ylabel("density")
fig.tight_layout()
plt.show()

print(f"Residuals for the {len(good):,} objects passing the cuts (median and 16-84% range):")
for label, (resid, _) in residuals.items():
    q16, q50, q84 = np.nanpercentile(resid, [16, 50, 84])
    print(f"  {label:>10s}: {q50:+.4f}  [{q16:+.4f}, {q84:+.4f}]")

## Next steps

* **Scale up:** set `N_SIM` to the full population (`n_total`, a few million), or run many independent seeds as separate batch jobs on a cluster and combine the output files.
* **Population:** add intrinsic scatter (see `intrinsic_scatter.ipynb`), hosts (`plasticc_snia.ipynb`), and Milky Way extinction.
* **Model mismatch:** fit with a different model than the one used to simulate (for example, simulate with `salt3-nir` and fit with `salt2-h17`) to test model systematics.
* **Survey:** add the deep drilling fields, or switch to a newer OpSim baseline by changing `OPSIM_FILE`.